# 004-reposed-frames — gallery

# Imports

In [ ]:
import os

os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

from uavloc.data.drone_frames import DroneFrame, load_frames
from uavloc.data.satellite import GeoImage
from uavloc.features.factory import build_features, correlator_config
from uavloc.features.segmentation import CLASSES
from uavloc.geo import LatLon
from uavloc.matching.correlator import PositionCorrelator
from uavloc.plotting.correlation import SEG_PALETTE, class_colours
from uavloc.track import REPO_ROOT, experiment_dir, load_runs

# Params

In [ ]:
EXP = "004-reposed-frames"
RUN = "004.2"
DEVICE = "mps"
ROWS_PER_FIGURE = 11
SAT_TOP_LEFT = (49.762965, 24.166472)
SAT_BOTTOM_RIGHT = (49.759291, 24.174046)

# Data

In [ ]:
config = next(r for r in load_runs(EXP) if r["run"] == RUN)["config"]
frames = load_frames(config["frames_dataset"])
satellite = GeoImage.load(REPO_ROOT / config["sat_image"], LatLon(*SAT_TOP_LEFT), LatLon(*SAT_BOTTOM_RIGHT))
len(frames)

# Model

In [ ]:
correlator = PositionCorrelator(build_features(config, satellite, DEVICE), correlator_config(config))

# Plots

In [ ]:
def gallery_row(axes: np.ndarray, frame: DroneFrame) -> None:
    """Drone frame | drone classes | orthophoto at the true pose | hand labels at the true pose."""
    folder = frame.image_path.parent
    label_ids = np.asarray(Image.open(folder / "label.png"))
    panels = [(frame.image(), "drone frame"),
              (class_colours(correlator.drone_map(frame), list(range(len(CLASSES)))), "drone segmentation (round 3)"),
              (Image.open(folder / "satellite.png"), "orthophoto at true pose"),
              (SEG_PALETTE[label_ids], "hand labels at true pose")]
    for ax, (image, title) in zip(axes, panels):
        ax.imshow(image); ax.set_title(title, fontsize=8); ax.set_xticks([]); ax.set_yticks([])
    axes[0].set_ylabel(f"{frame.name[-7:]}\nalt {frame.altitude_m:.0f} m\nyaw {frame.yaw_deg:.0f}", fontsize=8)

In [ ]:
pages = [frames[i:i + ROWS_PER_FIGURE] for i in range(0, len(frames), ROWS_PER_FIGURE)]
handles = [plt.Rectangle((0, 0), 1, 1, color=SEG_PALETTE[c] / 255) for c in range(len(CLASSES))]
figures = []
for page in pages:
    fig, axes = plt.subplots(len(page), 4, figsize=(16, 2.4 * len(page)))
    for row, frame in zip(axes, page):
        gallery_row(row, frame)
    fig.legend(handles, CLASSES, loc="lower center", ncol=len(CLASSES))
    fig.tight_layout(rect=(0, .02, 1, 1))
    figures.append(fig)

In [ ]:
out = experiment_dir(EXP) / "figures" / "gallery"
out.mkdir(parents=True, exist_ok=True)
for number, fig in enumerate(figures, 1):
    fig.savefig(out / f"gallery-{number}.png", dpi=80)
sorted(p.name for p in out.iterdir())